In [10]:
import pandas as pd
import numpy as np


In [11]:
data=pd.read_csv(r"C:\Users\Nada Abdulbaky\OneDrive\Desktop\NeuralNetwork (2)\NeuralNetwork\NeuralNetwork\Task1\data\penguins.csv")

In [12]:
# Display first 5 samples
print(data.head())

# Display dataset information
print(data.info())

# Check missing values
print(data.isnull().sum())


  Species  CulmenLength  CulmenDepth  FlipperLength OriginLocation  BodyMass
0  Adelie          39.1         18.7          181.0      Torgersen    3750.0
1  Adelie          39.5         17.4          186.0      Torgersen    3800.0
2  Adelie          40.3         18.0          195.0      Torgersen    3250.0
3  Adelie           NaN          NaN            NaN      Torgersen       NaN
4  Adelie          36.7         19.3          193.0      Torgersen    3450.0
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Species         150 non-null    str    
 1   CulmenLength    148 non-null    float64
 2   CulmenDepth     148 non-null    float64
 3   FlipperLength   148 non-null    float64
 4   OriginLocation  150 non-null    str    
 5   BodyMass        148 non-null    float64
dtypes: float64(4), str(2)
memory usage: 9.1 KB
None
Species           0
CulmenLength

In [13]:

# Check the number of samples in each class
print(data["Species"].value_counts())

Species
Adelie       50
Chinstrap    50
Gentoo       50
Name: count, dtype: int64


In [14]:
train_data = []
test_data = []

for species in data["Species"].unique():
    class_data = data[data["Species"] == species]

    train_data.append(class_data.iloc[:30])
    test_data.append(class_data.iloc[30:50])

train_data = pd.concat(train_data)
test_data = pd.concat(test_data)


In [15]:
train_data = train_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [16]:
print(train_data.head())

     Species  CulmenLength  CulmenDepth  FlipperLength OriginLocation  \
0  Chinstrap          46.6         17.8          193.0          Dream   
1     Adelie          35.9         19.2          189.0         Biscoe   
2  Chinstrap          52.0         19.0          197.0          Dream   
3     Gentoo          40.9         13.7          214.0         Biscoe   
4     Adelie          39.1         18.7          181.0      Torgersen   

   BodyMass  
0    3800.0  
1    3800.0  
2    4150.0  
3    4650.0  
4    3750.0  


In [17]:
features = [
    "CulmenLength",
    "CulmenDepth",
    "FlipperLength",
    "BodyMass",
    "OriginLocation"
]

X_train = train_data[features]
y_train = train_data["Species"]

X_test = test_data[features]
y_test = test_data["Species"]

In [18]:
numeric_features = [
    "CulmenLength",
    "CulmenDepth",
    "FlipperLength",
    "BodyMass"
]

meds = X_train[numeric_features].median()

X_train[numeric_features] = X_train[numeric_features].fillna(meds)
X_test[numeric_features] = X_test[numeric_features].fillna(meds)

In [19]:
# Encode OriginLocation as one numerical feature

location_map = {
    "Biscoe": 0,
    "Dream": 1,
    "Torgersen": 2
}

X_train["OriginLocation"] = (
    X_train["OriginLocation"].map(location_map)
)

X_test["OriginLocation"] = (
    X_test["OriginLocation"].map(location_map)
)

print("Training columns:", X_train.columns.tolist())
print("Testing columns:", X_test.columns.tolist())

Training columns: ['CulmenLength', 'CulmenDepth', 'FlipperLength', 'BodyMass', 'OriginLocation']
Testing columns: ['CulmenLength', 'CulmenDepth', 'FlipperLength', 'BodyMass', 'OriginLocation']


In [20]:
print(X_train.head())

   CulmenLength  CulmenDepth  FlipperLength  BodyMass  OriginLocation
0          46.6         17.8          193.0    3800.0               1
1          35.9         19.2          189.0    3800.0               0
2          52.0         19.0          197.0    4150.0               1
3          40.9         13.7          214.0    4650.0               0
4          39.1         18.7          181.0    3750.0               2


In [21]:
mean = X_train.mean()
std = X_train.std()

std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [22]:
print(y_train.head())

0    Chinstrap
1       Adelie
2    Chinstrap
3       Gentoo
4       Adelie
Name: Species, dtype: str


In [23]:
print(type(y_train))
print(y_train.head())
print(type(y_test))
print(y_test.head())

<class 'pandas.Series'>
0    Chinstrap
1       Adelie
2    Chinstrap
3       Gentoo
4       Adelie
Name: Species, dtype: str
<class 'pandas.Series'>
30    Adelie
31    Adelie
32    Adelie
33    Adelie
34    Adelie
Name: Species, dtype: str


In [24]:
# One-Hot Encoding for target labels

y_train = pd.get_dummies(y_train, dtype=int)
y_test = pd.get_dummies(y_test, dtype=int)

y_test = y_test.reindex(
    columns=y_train.columns,
    fill_value=0
)

In [25]:
# Convert features to NumPy arrays

X_train = X_train.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)

# Get class names before converting labels
class_names = y_train.columns.tolist()

# Convert one-hot labels to class indices
y_train = np.argmax(y_train.to_numpy(), axis=1)
y_test = np.argmax(y_test.to_numpy(), axis=1)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Number of input features:", X_train.shape[1])
print("Number of classes:", len(class_names))
print("Classes:", class_names)

Training shape: (90, 5)
Testing shape: (60, 5)
Number of input features: 5
Number of classes: 3
Classes: ['Adelie', 'Chinstrap', 'Gentoo']


In [26]:
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(a):
    return a * (1 - a)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(a):
    return 1 - a ** 2

In [27]:
class MLP:

    def __init__(self, input_size, hidden_layers, output_size,
                 learning_rate, activation, use_bias):

        self.learning_rate = learning_rate
        self.use_bias = use_bias
        self.activation_name = activation

        if activation == "sigmoid":
            self.activation = sigmoid
            self.activation_derivative = sigmoid_derivative
        else:
            self.activation = tanh
            self.activation_derivative = tanh_derivative

        self.layers = [input_size] + hidden_layers + [output_size]

        np.random.seed(42)

        self.weights = []
        self.biases = []

        for i in range(len(self.layers) - 1):

            w = np.random.uniform(-0.1, 0.1,(self.layers[i], self.layers[i + 1]))

            b = np.random.uniform(-0.1, 0.1,(1, self.layers[i + 1]))

            self.weights.append(w)
            self.biases.append(b)


    def forward(self, x):

        self.activations = [x.reshape(1, -1)]

        current = self.activations[0]

        for i in range(len(self.weights)):

            z = np.dot(current, self.weights[i])

            if self.use_bias:
                z += self.biases[i]

            current = self.activation(z)

            self.activations.append(current)

        return current


    def backward(self, target):

        output = self.activations[-1]

        error = target - output

        delta = error * self.activation_derivative(output)

        for i in reversed(range(len(self.weights))):

            previous_activation = self.activations[i]
            if i > 0:

                previous_delta = (
                    np.dot(delta, self.weights[i].T)
                    * self.activation_derivative(self.activations[i])
                )

            weight_gradient = np.dot(
                previous_activation.T, delta
            )

            self.weights[i] += (self.learning_rate * weight_gradient)

            if self.use_bias:
                self.biases[i] += self.learning_rate * delta

            if i > 0:
                delta = previous_delta


    def train(self, X, y, epochs):

        for epoch in range(epochs):

            for i in range(len(X)):

                output = self.forward(X[i])

                target = np.zeros((1, len(class_names)))

                if self.activation_name == "sigmoid":
                    target[0, y[i]] = 1

                else:
                    target.fill(-1)
                    target[0, y[i]] = 1

                self.backward(target)

            if (epoch + 1) % max(1, epochs // 10) == 0:

                predictions = self.predict(X)

                accuracy = np.mean(predictions == y) * 100

                print(
                    f"Epoch {epoch + 1}/{epochs} "
                    f"- Training Accuracy: {accuracy:.2f}%"
                )


    def predict_sample(self, x):

        output = self.forward(x)

        return np.argmax(output)


    def predict(self, X):

        predictions = []

        for sample in X:
            predictions.append(self.predict_sample(sample))

        return np.array(predictions)

In [28]:

num_hidden_layers = int(input("Enter number of hidden layers: "))

hidden_layers = []

for i in range(num_hidden_layers):

    neurons = int(input(f"Enter number of neurons in hidden layer {i + 1}: "))
    hidden_layers.append(neurons)

learning_rate = float(input("Enter learning rate (eta): "))
epochs = int(input("Enter number of epochs: "))
bias_choice = input("Use bias? (yes/no): ").lower()

while bias_choice not in ["yes", "no"]:
    bias_choice = input("Please enter yes or no: ").lower()

use_bias = bias_choice == "yes"

activation_name = input("Choose activation function (sigmoid/tanh): ").lower()

while activation_name not in ["sigmoid", "tanh"]:
    activation_name = input("Please enter sigmoid or tanh: ").lower()


print("\nNetwork Architecture:")
print([X_train.shape[1]] + hidden_layers + [len(class_names)])


Network Architecture:
[5, 5, 3]


In [29]:
model = MLP(
    input_size=X_train.shape[1],
    hidden_layers=hidden_layers,
    output_size=len(class_names),
    learning_rate=learning_rate,
    activation=activation_name,
    use_bias=use_bias
)

model.train(X_train, y_train, epochs)

Epoch 50/500 - Training Accuracy: 96.67%
Epoch 100/500 - Training Accuracy: 97.78%
Epoch 150/500 - Training Accuracy: 97.78%
Epoch 200/500 - Training Accuracy: 97.78%
Epoch 250/500 - Training Accuracy: 97.78%
Epoch 300/500 - Training Accuracy: 98.89%
Epoch 350/500 - Training Accuracy: 98.89%
Epoch 400/500 - Training Accuracy: 100.00%
Epoch 450/500 - Training Accuracy: 100.00%
Epoch 500/500 - Training Accuracy: 100.00%


In [30]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)
train_accuracy = np.mean(train_predictions == y_train) * 100
test_accuracy = np.mean(test_predictions == y_test) * 100

print(f"Training Accuracy: {train_accuracy:.2f}%")
print(f"Testing Accuracy: {test_accuracy:.2f}%")

Training Accuracy: 100.00%
Testing Accuracy: 95.00%


In [31]:
def calculate_confusion_matrix(y_true, y_pred, num_classes):

    matrix = np.zeros(
        (num_classes, num_classes),
        dtype=int
    )

    for actual, predicted in zip(y_true, y_pred):

        matrix[actual][predicted] += 1

    return matrix


# Calculate confusion matrix using test predictions
conf_matrix = calculate_confusion_matrix(
    y_test,
    test_predictions,
    len(class_names)
)


confusion_df = pd.DataFrame(
    conf_matrix,
    index=class_names,
    columns=class_names
)

print("Confusion Matrix:")
print(confusion_df)

Confusion Matrix:
           Adelie  Chinstrap  Gentoo
Adelie         20          0       0
Chinstrap       1         19       0
Gentoo          0          2      18


In [32]:
print("Predicted class distribution:")
for i, name in enumerate(class_names):
    print(name, ":", np.sum(test_predictions == i))

print("\nActual class distribution:")
for i, name in enumerate(class_names):
    print(name, ":", np.sum(y_test == i))

print("\nConfusion Matrix:")
print(confusion_df)

Predicted class distribution:
Adelie : 21
Chinstrap : 21
Gentoo : 18

Actual class distribution:
Adelie : 20
Chinstrap : 20
Gentoo : 20

Confusion Matrix:
           Adelie  Chinstrap  Gentoo
Adelie         20          0       0
Chinstrap       1         19       0
Gentoo          0          2      18


In [33]:
print("Training labels distribution:")
print(np.unique(y_train, return_counts=True))

print("\nTesting labels distribution:")
print(np.unique(y_test, return_counts=True))


print("\nTraining predictions distribution:")
print(np.unique(train_predictions, return_counts=True))


print("\nTesting predictions distribution:")
print(np.unique(test_predictions, return_counts=True))

Training labels distribution:
(array([0, 1, 2]), array([30, 30, 30]))

Testing labels distribution:
(array([0, 1, 2]), array([20, 20, 20]))

Training predictions distribution:
(array([0, 1, 2]), array([30, 30, 30]))

Testing predictions distribution:
(array([0, 1, 2]), array([21, 21, 18]))


In [34]:
for i in range(3):
    sample = X_test[i]
    output = model.forward(sample)

    print("Actual class:", class_names[y_test[i]])
    print("Output:", output)
    print("Predicted class:", class_names[np.argmax(output)])
    print()

Actual class: Adelie
Output: [[0.90604883 0.14704163 0.00475284]]
Predicted class: Adelie

Actual class: Adelie
Output: [[0.99107032 0.00760366 0.0071254 ]]
Predicted class: Adelie

Actual class: Adelie
Output: [[0.97973019 0.02868707 0.00379126]]
Predicted class: Adelie



In [35]:
print("NaN values in X_train:", np.isnan(X_train).sum())
print("NaN values in X_test:", np.isnan(X_test).sum())

print("Infinite values in X_train:", np.isinf(X_train).sum())
print("Infinite values in X_test:", np.isinf(X_test).sum())

print("\nTraining data finite:", np.isfinite(X_train).all())
print("Testing data finite:", np.isfinite(X_test).all())

NaN values in X_train: 0
NaN values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0

Training data finite: True
Testing data finite: True


In [36]:
for i in range(len(model.weights)):
    print(f"Layer {i + 1}")

    print("Weights finite:",
          np.isfinite(model.weights[i]).all())

    print("Biases finite:",
          np.isfinite(model.biases[i]).all())

    print("Max absolute weight:",
          np.max(np.abs(model.weights[i])))

Layer 1
Weights finite: True
Biases finite: True
Max absolute weight: 3.5034031448762604
Layer 2
Weights finite: True
Biases finite: True
Max absolute weight: 4.247519593661175


In [37]:
print("Training feature statistics:")
print(pd.DataFrame(X_train).describe())

print("\nTesting feature statistics:")
print(pd.DataFrame(X_test).describe())

Training feature statistics:
                  0             1             2             3             4
count  9.000000e+01  9.000000e+01  9.000000e+01  9.000000e+01  9.000000e+01
mean  -8.499374e-16  7.288923e-16 -5.933525e-16 -1.159566e-16 -8.018277e-18
std    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
min   -2.067464e+00 -1.931804e+00 -1.966551e+00 -1.630235e+00 -9.844344e-01
25%   -8.362200e-01 -9.564673e-01 -7.545489e-01 -7.168652e-01 -9.844344e-01
50%    2.411186e-01  2.254111e-01 -2.511019e-01 -3.369680e-01  2.812670e-01
75%    7.845975e-01  7.302912e-01  9.236076e-01  5.521533e-01  2.812670e-01
max    2.530463e+00  1.923644e+00  2.359364e+00  2.766873e+00  1.546968e+00

Testing feature statistics:
               0          1          2          3          4
count  60.000000  60.000000  60.000000  60.000000  60.000000
mean    0.123445   0.085422   0.159114   0.071489  -0.140633
std     1.076145   0.861393   1.121969   1.130394   0.601693
min    -1.7019

In [38]:
print("Training Confusion Matrix:")

train_conf_matrix = calculate_confusion_matrix(
    y_train,
    train_predictions,
    len(class_names)
)

print(pd.DataFrame(
    train_conf_matrix,
    index=class_names,
    columns=class_names
))

Training Confusion Matrix:
           Adelie  Chinstrap  Gentoo
Adelie         30          0       0
Chinstrap       0         30       0
Gentoo          0          0      30
